# SQLite и модуль `sqlite3` в Python  

В этой работе вы создадите небольшую библиотечную базу данных, научитесь добавлять, искать, изменять и удалять записи, а затем реализуете учёт выдачи книг и отчёты.

**Как выполнять работу**

- Все запросы SQL, функции Python и результаты их проверки размещайте **непосредственно в ячейках этого ноутбука**.
- Работайте с одной базой данных `library.db`. Её таблицы будут изменяться по мере выполнения заданий.

## Подготовка

Импортируйте модуль `sqlite3`. Используйте указанное имя базы данных во всех заданиях. При желании добавляйте собственные ячейки для экспериментов и проверки.

In [1]:
import sqlite3

DB_PATH = "library.db"

---
## Задача 1. «Знакомство»

Создайте новую базу данных **`library.db`**. Внутри базы создайте таблицу **`books`** со следующими полями:

| Поле | Тип и ограничение |
|---|---|
| `id` | `INTEGER PRIMARY KEY` |
| `title` | `TEXT NOT NULL` |
| `author` | `TEXT NOT NULL` |
| `year` | `INTEGER` |

Напишите код для создания таблицы в ячейке ниже. Затем проверьте, что таблица существует и её поля заданы верно.

In [2]:
# Задача 1. Создание базы данных и таблицы books
# Ваш код
db = sqlite3.connect(DB_PATH)
cursor = db.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS books(id INTEGER PRIMARY KEY, title TEXT NOT NULL, author TEXT NOT NULL, year INTEGER)")
db.commit()

In [3]:
# Задача 1. Проверка структуры таблицы books
# Ваш код
cursor.execute("SELECT * FROM books;")
print(cursor.fetchall())
cursor.close()

[]


---
## Задача 2. «Пополнение полки»

Добавьте книги в таблицу `books` **двумя способами**:

1. Выполните **пять отдельных запросов `INSERT`**, добавив пять различных книг.
2. Используйте **один параметризованный шаблон `INSERT`** с заполнителями `?` и список из пяти кортежей; передайте их с помощью `executemany()`.

Используйте **разные книги для каждого способа**, чтобы всего в таблице оказалось не менее 10 записей. Подберите книги разных лет: в последующих заданиях понадобятся как издания **до 2000 года**, так и издания **2000 года и позже**.

После вставки выведите все записи и убедитесь, что данные сохранились.

In [ ]:
# Задача 2, способ 1. Пять отдельных INSERT
# Ваш код

booksData = [
    { 'title': 'Fst book', 'author': 'Sashaprok', 'year': 2006 },
    { 'title': '2nd book', 'author': 'Sashaprok1', 'year': 1998 },
    { 'title': '3rd book', 'author': 'Sashaprok2', 'year': 2010 },
    { 'title': '4 book', 'author': 'Sashaprok3', 'year': 2016 },
    { 'title': '5 book', 'author': 'Sashaprok4', 'year': 2106 },
]

cursor = db.cursor()
for i, book in enumerate(booksData):
    cursor.execute("INSERT INTO books(title, author, year) VALUES (?, ?, ?)", (book['title'], book['author'], book['year']))
cursor.fetchall()
db.commit()
cursor.close()

In [ ]:
# Задача 2, способ 2. Параметризованный INSERT + executemany()
cursor = db.cursor()
data = [(book['title'], book['author'], book['year']) for book in booksData]
cursor.executemany("INSERT INTO books(title, author, year) VALUES (?, ?, ?)", data)
cursor.fetchone()
db.commit()
cursor.close()
# Ваш код

In [12]:
# Задача 2. Проверка добавленных книг
# Ваш код
cursor = db.cursor()
cursor.execute("SELECT * FROM books")
print(cursor.fetchall())
cursor.close()

[(0, 'Fst book', 'Sashaprok', 2006), (1, '2nd book', 'Sashaprok1', 1998), (2, '3rd book', 'Sashaprok2', 2010), (3, '4 book', 'Sashaprok3', 2016), (4, '5 book', 'Sashaprok4', 2106), (5, 'Fst book', 'Sashaprok', 2006), (6, '2nd book', 'Sashaprok1', 1998), (7, '3rd book', 'Sashaprok2', 2010), (8, '4 book', 'Sashaprok3', 2016), (9, '5 book', 'Sashaprok4', 2106), (10, 'Fst book', 'Sashaprok', 2006), (11, '2nd book', 'Sashaprok1', 1998), (12, '3rd book', 'Sashaprok2', 2010), (13, '4 book', 'Sashaprok3', 2016), (14, '5 book', 'Sashaprok4', 2106)]


---
## Задача 3. «Каталог»

Напишите функцию **`get_books(n=None)`**, которая извлекает записи из таблицы `books`:

- Если `n is None`, функция возвращает **все** книги.
- Если `n` — целое число, функция возвращает **первые `n` записей**.
- Результат должен быть **списком кортежей**.

Для однозначного порядка считайте «первыми» записи с наименьшими `id` (сортировка `ORDER BY id`).

**Пример формата результата:**

```python
[
    (1, 'Война и мир', 'Лев Толстой', 1869),
    (2, 'Преступление и наказание', 'Фёдор Достоевский', 1866),
    (3, '1984', 'Джордж Оруэлл', 1949)
]
```

Реальные значения зависят от книг, добавленных вами в задаче 2.

In [56]:
# Задача 3. Реализуйте функцию get_books

def get_books(n=None):
    cursor = db.cursor()
    cursor.execute("SELECT * FROM books")
    r = cursor.fetchmany(n or -1)
    cursor.close()
    return r

In [57]:
# Задача 3. Проверьте работу с n=None и с целым n
# Ваш код
print(get_books(1))
print(get_books())

[(0, 'Fst book', 'Sashaprok', 2006)]
[(0, 'Fst book', 'Sashaprok', 2006), (1, '2nd book', 'Sashaprok1', 1998), (2, '3rd book', 'Sashaprok2', 2010), (3, '4 book', 'Sashaprok3', 2016), (4, '5 book', 'Sashaprok4', 2106), (5, 'Fst book', 'Sashaprok', 2006), (6, '2nd book', 'Sashaprok1', 1998), (7, '3rd book', 'Sashaprok2', 2010), (8, '4 book', 'Sashaprok3', 2016), (9, '5 book', 'Sashaprok4', 2106), (10, 'Fst book', 'Sashaprok', 2006), (11, '2nd book', 'Sashaprok1', 1998), (12, '3rd book', 'Sashaprok2', 2010), (13, '4 book', 'Sashaprok3', 2016), (14, '5 book', 'Sashaprok4', 2106)]


---
## Задача 4. «Поиск по жанру и автору»

**Часть 1.** Измените структуру таблицы `books`: добавьте столбец **`genre TEXT`**. Затем обновите существующие записи, указав для каждой книги жанр (например, «Фантастика», «Детектив», «Роман»).

**Часть 2.** Напишите функцию **`find_books(genre=None, author=None)`**, которая возвращает книги:

- по указанному жанру, если передан только `genre`;
- по указанному автору, если передан только `author`;
- по обоим условиям одновременно, если переданы `genre` и `author`;
- все книги, если аргументы не переданы.

Функция должна возвращать **список кортежей**. Проверьте все четыре случая.

In [ ]:
# Задача 4, часть 1. Добавьте столбец genre и заполните его
# Ваш код

cursor = db.cursor()
cursor.execute("ALTER TABLE books ADD COLUMN genre TEXT")
cursor.execute("UPDATE books SET genre='Fantasy'")
cursor.fetchone()
cursor.close()
db.commit()

In [67]:
# Задача 4, часть 2. Реализуйте функцию find_books

def find_books(genre=None, author=None):
    cursor = db.cursor()
    cursor.execute("SELECT * FROM books WHERE (:genre IS NULL OR genre = :genre) AND (:author IS NULL OR author = :author)", {"genre": genre, "author": author})
    r = cursor.fetchall()
    cursor.close()
    return r
    raise NotImplementedError

In [68]:
# Задача 4. Проверьте поиск по жанру, автору, их комбинации и без фильтров
# Ваш код
find_books(author="Sashaprok")

[(0, 'Fst book', 'Sashaprok', 2006, 'Fantasy'),
 (5, 'Fst book', 'Sashaprok', 2006, 'Fantasy'),
 (10, 'Fst book', 'Sashaprok', 2006, 'Fantasy')]

---
## Задача 5. «Смена владельца»

Добавьте возможность **«взятия книги на чтение»**.

1. Добавьте в таблицу `books` поле **`is_available BOOLEAN DEFAULT TRUE`**.
2. Напишите функцию **`borrow_book(book_id)`**, которая устанавливает `is_available = FALSE` для книги с указанным `id`.
3. Напишите функцию **`return_book(book_id)`**, которая устанавливает `is_available = TRUE` для книги с указанным `id`.

Проверьте работу функций на одной из книг: выведите её статус до выдачи, после выдачи и после возврата. Не забудьте сохранять изменения в базе.


In [69]:
# Задача 5, часть 1. Добавьте столбец is_available
# Ваш код
cursor = db.cursor()
cursor.execute("ALTER TABLE books ADD COLUMN is_available BOOLEAN DEFAULT TRUE")
cursor.fetchone()
cursor.close()
db.commit()

In [70]:
# Задача 5, части 2–3. Реализуйте функции

def borrow_book(book_id):
    cursor = db.cursor()
    cursor.execute("UPDATE books SET is_available = FALSE WHERE id = ?", [book_id])
    cursor.fetchall()
    cursor.close()


def return_book(book_id):
    cursor = db.cursor()
    cursor.execute("UPDATE books SET is_available = TRUE WHERE id = ?", [book_id])
    cursor.fetchall()
    cursor.close()

In [71]:
# Задача 5. Проверьте выдачу и возврат одной книги
# Ваш код

borrow_book(1)
return_book(1)

---
## Задача 6. «Списание старого фонда»

Удалите из таблицы `books` все книги, **изданные раньше 2000 года**.

Перед удалением обязательно:

1. Выведите на экран список книг, которые будут удалены.
2. Проверьте этот список и убедитесь, что именно эти записи нужно удалить.
3. Удалите выбранные записи и сохраните изменения.
4. Выведите список оставшихся книг, чтобы убедиться в результате.

**Условие:** книги, изданные **в 2000 году**, удалять не нужно.

> Перед выполнением убедитесь, что в таблице останутся книги: они понадобятся для заданий 7–8.

In [73]:
# Задача 6. Просмотр списка и удаление книг до 2000 года
# Ваш код
cursor = db.cursor()
cursor.execute("SELECT * FROM books WHERE year < 2000")
print(cursor.fetchall())
cursor.execute("DELETE FROM books WHRER year < 2000")
cursor.execute("SELECT * FROM books")
print(cursor.fetchall())
cursor.close()

[(1, '2nd book', 'Sashaprok1', 1998, 'Fantasy', 1), (6, '2nd book', 'Sashaprok1', 1998, 'Fantasy', 1), (11, '2nd book', 'Sashaprok1', 1998, 'Fantasy', 1)]


OperationalError: near "WHRER": syntax error

---
## Задача 7. «Система читателей»

Создайте таблицу **`readers`**:

| Поле | Тип и ограничение |
|---|---|
| `id` | `INTEGER PRIMARY KEY` |
| `name` | `TEXT NOT NULL` |

Создайте таблицу **`records`** для учёта выдач книг:

| Поле | Тип и ограничение |
|---|---|
| `id` | `INTEGER PRIMARY KEY` |
| `book_id` | `INTEGER NOT NULL` |
| `reader_id` | `INTEGER NOT NULL` |
| `borrow_date` | `TEXT DEFAULT CURRENT_DATE` |
| `return_date` | `TEXT`, может быть `NULL` |

Добавьте **несколько читателей** в `readers`.

Затем реализуйте функции:

- **`borrow_book_to_reader(book_id, reader_id)`** — добавляет в `records` запись о выдаче книги читателю; поле `return_date` остаётся `NULL`.
- **`return_book_from_reader(record_id)`** — отмечает возврат книги, проставляя дату в `return_date` соответствующей записи.

Используйте текущую дату SQLite (`CURRENT_DATE`) либо дату Python в формате `YYYY-MM-DD`. Для согласованности данных рекомендуется также обновлять `books.is_available` при выдаче и возврате.

**Проверка:** выдайте книги нескольким читателям, одну из книг верните, затем выведите содержимое `records`. Не удаляйте историю выдач: она пригодится для задания 8.

In [ ]:
# Задача 7, часть 1. Создайте таблицы readers и records
# Ваш код

In [ ]:
# Задача 7, часть 2. Добавьте несколько читателей
# Ваш код

In [ ]:
# Задача 7, часть 3. Реализуйте функции учёта выдачи и возврата

def borrow_book_to_reader(book_id, reader_id):
    # Ваш код
    raise NotImplementedError


def return_book_from_reader(record_id):
    # Ваш код
    raise NotImplementedError

In [ ]:
# Задача 7. Выполните тестовые выдачи и возвраты, проверьте таблицу records
# Ваш код

---
## Задача 8. «Отчёты для библиотекаря»

Напишите три функции:

1. **`borrowed_books()`** — находит книги, которые сейчас на руках (`return_date IS NULL`), и показывает названия книг **вместе с именами читателей**, которым они выданы.
2. **`reader_stats()`** — подсчитывает количество книг, находящихся на руках **у каждого читателя в данный момент**. Включите в отчёт и читателей, у которых сейчас нет книг (количество `0`).
3. **`popular_books()`** — находит **три самые популярные книги** по общему количеству выдач (включая уже возвращённые книги).

Используйте `JOIN`, агрегатные функции, `GROUP BY`, `ORDER BY` и `LIMIT`, где это необходимо.

Функции должны возвращать данные в удобном для дальнейшего использования виде (например, **списки кортежей**). После реализации вызовите все три функции и выведите результаты.

> Для содержательной проверки в базе должны быть несколько записей о выдаче, в том числе о возвращённых и ещё не возвращённых книгах.

In [ ]:
# Задача 8. Реализуйте три функции для отчётов

def borrowed_books():
    # Ваш код
    raise NotImplementedError


def reader_stats():
    # Ваш код
    raise NotImplementedError


def popular_books():
    # Ваш код
    raise NotImplementedError

In [ ]:
# Задача 8. Вызовите borrowed_books(), reader_stats() и popular_books()
# Ваш код

Сдайте заполненный ноутбук `.ipynb`. и сформированную базу `library.db`.